In [1]:
import os, sys, re
import pandas as pd
from sklearn.metrics import confusion_matrix

In [2]:
pd.set_option('display.max_colwidth', None)

In [3]:
DATA_FILE = "../results/samples_for_simple_sentences_results_analyzed.csv"

In [4]:
df = pd.read_csv(DATA_FILE, sep=",", encoding="utf-8")

# A ja S n80 
n80 = df[(df["initial_cat"]=="a_n80") | (df["initial_cat"]=="s_n80")]

# A ja S n80, mis ei ole ELT
n80_noelt = n80[(n80["tag"]!= "E") & (n80["tag"]!= "L") & (n80["tag"]!= "T")  & ~(n80["tags"].str.contains("ELT", na=False))]
n80_noelt = n80_noelt[(~n80_noelt["verb_head_same_sent"].isna()) & (~n80_noelt["head_verbi_alluv"].isna())]

n80_noelt = n80_noelt[['sentence_id', 'head_id', 'head_loc', 'verb', 'verb_compound',
       'morph_case', 'lemma', 'form', 'sentence', 'simple_sentences_gpt4o', 'tags', 'tag',
       'initial_cat', 'verb_head_same_sent', 'head_verbi_alluv']]

# A ja S n80, mis on ELT
n80_elt = n80[(n80["tag"]== "E") | (n80["tag"]== "L") | (n80["tag"]== "T")  | (n80["tags"].str.contains("ELT", na=False))]
#n80_elt = n80[(n80["tag"]!= "A") & (n80["tag"]!= "S") & ~(n80["tags"].str.contains("AS", na=False))]
n80_elt = n80_elt[(~n80_elt["verb_head_same_sent"].isna()) & (~n80_elt["head_verbi_alluv"].isna())]

n80_elt = n80_elt[['sentence_id', 'head_id', 'head_loc', 'verb', 'verb_compound',
       'morph_case', 'lemma', 'form', 'sentence', 'simple_sentences_gpt4o', 'tags', 'tag',
       'initial_cat', 'verb_head_same_sent', 'head_verbi_alluv']]


# Kokkuvõte A ja S n80, mis ei ole E/L/T (100 näidet)

In [5]:
same_sent = n80_noelt[n80_noelt["verb_head_same_sent"]=="y"]
not_same_sent = n80_noelt[n80_noelt["verb_head_same_sent"]=="n"]
not_same_but_error = not_same_sent[not_same_sent["head_verbi_alluv"]=="?"]
not_same_not_alluv = not_same_sent[not_same_sent["head_verbi_alluv"]=="n"]
not_same_orig_error =  not_same_sent[not_same_sent["head_verbi_alluv"]=="e"]
same_sent_alluv = same_sent[same_sent["head_verbi_alluv"]=="y"]
same_sent_replaced = same_sent[same_sent["head_verbi_alluv"]=="?"]
same_sent_kesksona = same_sent[same_sent["head_verbi_alluv"]=="k"]
same_sent_error = same_sent[same_sent["head_verbi_alluv"]=="e"]
#errors = n80_noelt[n80_noelt["verb_head_same_sent"]=="e"]
gpt_errors = n80_noelt[n80_noelt["verb_head_same_sent"]=="?"]

In [6]:
print("verb + peasõna on samas lihtlauses: ", len(same_sent))
print("verb+ps samas lihtlauses ja ps on verbi alluv: ", len(same_sent_alluv), "/", len(same_sent))
print("verb+ps samas lihtlauses aga peasõna on asendatud teise sõnaga (enamasti sisuliselt õige): ", len(same_sent_replaced), "/", len(same_sent))
print("verb+ps samas lihtlauses aga orig lauses kesksõna vorm: ", len(same_sent_kesksona), "/", len(same_sent))
print("verb+ps samas lihtlauses aga lauses probleem: ", len(same_sent_error), "/", len(same_sent))


print("\n")

print("verb + peasõna pole samas lihtlauses: ", len(not_same_sent))
print("verb+ps pole samas lihtlauses aga orig lauses on ps verbi alluv (gpt lausestamise viga):", len(not_same_but_error), "/", len(not_same_sent) )
print("verb+ps pole samas lihtlauses ja ps ei ole orig lauses verbi alluv:", len(not_same_not_alluv), "/", len(not_same_sent) )
print("verb+ps pole samas lihtlauses ja orig lauses midagi valesti:", len(not_same_orig_error), "/", len(not_same_sent) )

print("\n")

#print("peasõna süntaksi viga:", len(errors))
print("gpt lausestamisega muutus struktuur liiga palju:", len(gpt_errors))


verb + peasõna on samas lihtlauses:  89
verb+ps samas lihtlauses ja ps on verbi alluv:  70 / 89
verb+ps samas lihtlauses aga peasõna on asendatud teise sõnaga (enamasti sisuliselt õige):  4 / 89
verb+ps samas lihtlauses aga orig lauses kesksõna vorm:  8 / 89
verb+ps samas lihtlauses aga lauses probleem:  7 / 89


verb + peasõna pole samas lihtlauses:  9
verb+ps pole samas lihtlauses aga orig lauses on ps verbi alluv (gpt lausestamise viga): 4 / 9
verb+ps pole samas lihtlauses ja ps ei ole orig lauses verbi alluv: 0 / 9
verb+ps pole samas lihtlauses ja orig lauses midagi valesti: 5 / 9


gpt lausestamisega muutus struktuur liiga palju: 2


### Segadusmaatriks

In [7]:
sm_noelt = n80_noelt[n80_noelt["verb_head_same_sent"]!= "?"]
sm_noelt.loc[
    (sm_noelt["verb_head_same_sent"] == "n") &
    (sm_noelt["head_verbi_alluv"] == "?"),
    "head_verbi_alluv"
] = "y"
sm_noelt = sm_noelt[(sm_noelt["head_verbi_alluv"]!= "?") & (sm_noelt["head_verbi_alluv"]!= "k")]

In [8]:
pd.crosstab(
    sm_noelt["verb_head_same_sent"],
    sm_noelt["head_verbi_alluv"],
    rownames=["SameSent"],
    colnames=["Alluv"]
)

Alluv,e,y
SameSent,,
n,5,4
y,7,70


### Näited

In [7]:
not_same_sent 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
235,4020416,6471584,20,lugema,NaN,all,valijamees,valijameestele,"Tehtud on Eesti poliitilises kultuuris pretsedenditu erakondlik kampaaniareis Pärnumaale , kus Rahvaliit presidendi hõlma varjust omavalitsustele ( loe : valijameestele ) rahamägesid ja eurojõgesid lubas .","['Tehtud on erakondlik kampaaniareis Pärnumaale.', 'See on pretsedenditu Eesti poliitilises kultuuris.', 'Rahvaliit lubas presidendi hõlma varjust omavalitsustele rahamägesid ja eurojõgesid.', 'Omavalitsused on valijamehed.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,a_n80,n,e
245,2342833,3753052,2,kompenseerima,NaN,all,seadus,seadusele,"Vastavalt seadusele kompenseerib kohalik omavalitsus investeeringud teedeehitusse , drenaaži ja tänavavalgutusse , juhul kui need on avalikus kasutuses või kui ei lepita kokku teisiti .","['Seadus sätestab kohaliku omavalitsuse kohustused.', 'Kohalik omavalitsus kompenseerib investeeringud teedeehitusse, drenaaži ja tänavavalgustusse.', 'Kompensatsioon toimub juhul, kui need on avalikus kasutuses.', 'Kompensatsioon toimub ka juhul, kui ei lepita kokku teisiti.']",NaN,NaN,a_n80,n,e
246,2970206,4764348,3,kütma,NaN,all,tema,neile,""" Kütsime neile sauna ka rõivistusse , "" muheles Aivar Pohlak .","['Aivar Pohlak muheles.', 'Aivar Pohlak ütles, et nad kütsid sauna rõivistusse.']",NaN,A,a_n80,n,?
256,2737810,4391255,13,lugema,NaN,all,nooruk,noorukile,"Tartus Võru tänaval kinnipeetud Audi 80 roolis istunud 1987 aastal sündinud pimedale noorukile luges kaarti kõrvalistuja , kel samuti puudus juhiluba .","['Politsei pidas Tartus Võru tänaval kinni Audi 80.', 'Audi 80 roolis istus 1987. aastal sündinud pime nooruk.', 'Kaarti luges kõrvalistuja.', 'Ka kõrvalistujal puudus juhiluba.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,a_n80,n,?
258,10820295,17329943,8,mõjuma,NaN,all,diplomaat,diplomaadile,“ Postimehe ” arvates mõjub Venemaa suursaadiku diplomaadile kohatu üliaktiivsus meedia õpetamisel Eesti-Vene niigi habrastele suhetele pärssivalt .,"['“Postimees” avaldas arvamuse.', 'Venemaa suursaadiku üliaktiivsus meedia õpetamisel on diplomaadile kohatu.', 'See mõjub Eesti-Vene niigi habrastele suhetele pärssivalt.']",NaN,A,a_n80,n,e
776,16334815,25281664,61,surema,NaN,adit,kanza-surm,kanza-surma,"niisiis , uurisin antud linki sealt lehelt veel linke ja otsisin veel täiendust ja sain päris palju teada selle asja kohta : esiteks final fantasy sulle teadmiseks et alkohol seejärel tubakas on palju ohtlikumad ja suuremat sõltuvust tekitavad asjad kui kanza ( kanep tekitab vähem sõltuvust kui kohvi ntx ) , kanepiga pole võimalik üledoosi kätte surra , keegi pole kanza-surma surnud , kanza ei tekita inimeses vastupidiselt ntx muudele narkodele ja alcole vägivallatunnet , pigem rahulikku ja mõnusa olemise ( nagu mõne pilsneri joomine ) , kanza on paljus mõttes kasulik ( saab teha köit , ravimina jnejne . )","['Uurisin antud linki.', 'Leidsin sealt lehelt veel linke.', 'Otsisin veel täiendust.', 'Sain päris palju teada selle asja kohta.', 'Final Fantasy, sulle teadmiseks, et alkohol ja tubakas on palju ohtlikumad kui kanep.', 'Kanep tekitab vähem sõltuvust kui kohv.', 'Kanepiga pole võimalik üledoosi kätte surra.', 'Keegi pole kanepitarbimise tõttu surnud.', 'Kanep ei tekita inimeses vägivallatunnet.', 'Kanep tekitab pigem rahuliku ja mõnusa olemise.', 'Kanep on paljus mõttes kasulik.', 'Kanepist saab teha köit.', 'Kanepit kasutatakse ravimina.']",NaN,NaN,s_n80,n,?
780,6879142,11062320,15,surema,NaN,adit,surr,surri,"Wimaks kiskusid nemmad temmal kõrri lõua alt wälja , kus ta sure walloga ärra surri .","['Nemmad kiskusid Wimaks temmal kõrri lõua alt välja.', 'Ta suri walloga.']",NaN,S,s_n80,n,e
788,5373915,8619271,9,haigestuma,NaN,adit,turist,turisti,"Rohkem kui kümme eelmisel nädalavahetusel Lätit külastanud Soome turisti on haigestunud salmonelloosi , teatas Läti keskkonnatervise kaitse keskus .","['Rohkem kui

In [8]:
not_same_but_error

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
246,2970206,4764348,3,kütma,NaN,all,tema,neile,""" Kütsime neile sauna ka rõivistusse , "" muheles Aivar Pohlak .","['Aivar Pohlak muheles.', 'Aivar Pohlak ütles, et nad kütsid sauna rõivistusse.']",NaN,A,a_n80,n,?
256,2737810,4391255,13,lugema,NaN,all,nooruk,noorukile,"Tartus Võru tänaval kinnipeetud Audi 80 roolis istunud 1987 aastal sündinud pimedale noorukile luges kaarti kõrvalistuja , kel samuti puudus juhiluba .","['Politsei pidas Tartus Võru tänaval kinni Audi 80.', 'Audi 80 roolis istus 1987. aastal sündinud pime nooruk.', 'Kaarti luges kõrvalistuja.', 'Ka kõrvalistujal puudus juhiluba.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,a_n80,n,?
776,16334815,25281664,61,surema,NaN,adit,kanza-surm,kanza-surma,"niisiis , uurisin antud linki sealt lehelt veel linke ja otsisin veel täiendust ja sain päris palju teada selle asja kohta : esiteks final fantasy sulle teadmiseks et alkohol seejärel tubakas on palju ohtlikumad ja suuremat sõltuvust tekitavad asjad kui kanza ( kanep tekitab vähem sõltuvust kui kohvi ntx ) , kanepiga pole võimalik üledoosi kätte surra , keegi pole kanza-surma surnud , kanza ei tekita inimeses vastupidiselt ntx muudele narkodele ja alcole vägivallatunnet , pigem rahulikku ja mõnusa olemise ( nagu mõne pilsneri joomine ) , kanza on paljus mõttes kasulik ( saab teha köit , ravimina jnejne . )","['Uurisin antud linki.', 'Leidsin sealt lehelt veel linke.', 'Otsisin veel täiendust.', 'Sain päris palju teada selle asja kohta.', 'Final Fantasy, sulle teadmiseks, et alkohol ja tubakas on palju ohtlikumad kui kanep.', 'Kanep tekitab vähem sõltuvust kui kohv.', 'Kanepiga pole võimalik üledoosi kätte surra.', 'Keegi pole kanepitarbimise tõttu surnud.', 'Kanep ei tekita inimeses vägivallatunnet.', 'Kanep tekitab pigem rahuliku ja mõnusa olemise.', 'Kanep on paljus mõttes kasulik.', 'Kanepist saab teha köit.', 'Kanepit kasutatakse ravimina.']",NaN,NaN,s_n80,n,?
791,1163886,1849246,9,surema,NaN,adit,kopsuvähk,kopsuvähki,"Nädal tagasi suri 61. eluaastal oma kodus Fairfaxis kopsuvähki Michaela Odone , kelle katsed leida ravimit oma poja raske päriliku haiguse vastu said aluseks Eestigi telekanaleil mitu korda näidatud filmile "" Lorenzo õli "" .","['Michaela Odone suri nädal tagasi.', 'Ta suri 61. eluaastal.', 'Ta suri oma kodus Fairfaxis.', 'Surma põhjus oli kopsuvähk.', 'Michaela Odone otsis ravimit oma poja raske päriliku haiguse vastu.', 'Need katsed said aluseks filmile ""Lorenzo õli"".', 'Filmi on näidatud ka Eesti telekanalitel.']",|S|AS|LS|ST|AST|LST|,NaN,s_n80,n,?


In [9]:
not_same_not_alluv

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv


In [10]:
not_same_orig_error

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
235,4020416,6471584,20,lugema,NaN,all,valijamees,valijameestele,"Tehtud on Eesti poliitilises kultuuris pretsedenditu erakondlik kampaaniareis Pärnumaale , kus Rahvaliit presidendi hõlma varjust omavalitsustele ( loe : valijameestele ) rahamägesid ja eurojõgesid lubas .","['Tehtud on erakondlik kampaaniareis Pärnumaale.', 'See on pretsedenditu Eesti poliitilises kultuuris.', 'Rahvaliit lubas presidendi hõlma varjust omavalitsustele rahamägesid ja eurojõgesid.', 'Omavalitsused on valijamehed.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,a_n80,n,e
245,2342833,3753052,2,kompenseerima,NaN,all,seadus,seadusele,"Vastavalt seadusele kompenseerib kohalik omavalitsus investeeringud teedeehitusse , drenaaži ja tänavavalgutusse , juhul kui need on avalikus kasutuses või kui ei lepita kokku teisiti .","['Seadus sätestab kohaliku omavalitsuse kohustused.', 'Kohalik omavalitsus kompenseerib investeeringud teedeehitusse, drenaaži ja tänavavalgustusse.', 'Kompensatsioon toimub juhul, kui need on avalikus kasutuses.', 'Kompensatsioon toimub ka juhul, kui ei lepita kokku teisiti.']",NaN,NaN,a_n80,n,e
258,10820295,17329943,8,mõjuma,NaN,all,diplomaat,diplomaadile,“ Postimehe ” arvates mõjub Venemaa suursaadiku diplomaadile kohatu üliaktiivsus meedia õpetamisel Eesti-Vene niigi habrastele suhetele pärssivalt .,"['“Postimees” avaldas arvamuse.', 'Venemaa suursaadiku üliaktiivsus meedia õpetamisel on diplomaadile kohatu.', 'See mõjub Eesti-Vene niigi habrastele suhetele pärssivalt.']",NaN,A,a_n80,n,e
780,6879142,11062320,15,surema,NaN,adit,surr,surri,"Wimaks kiskusid nemmad temmal kõrri lõua alt wälja , kus ta sure walloga ärra surri .","['Nemmad kiskusid Wimaks temmal kõrri lõua alt välja.', 'Ta suri walloga.']",NaN,S,s_n80,n,e
788,5373915,8619271,9,haigestuma,NaN,adit,turist,turisti,"Rohkem kui kümme eelmisel nädalavahetusel Lätit külastanud Soome turisti on haigestunud salmonelloosi , teatas Läti keskkonnatervise kaitse keskus .","['Rohkem kui kümme Soome turisti külastas eelmisel nädalavahetusel Lätit.', 'Need turistid on haigestunud salmonelloosi.', 'Läti keskkonnatervise kaitse keskus teatas sellest.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,s_n80,n,e


In [11]:
same_sent.sample(5)

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
206,13862539,22104496,13,põhjustama,NaN,all,suusaliit,suusaliidule,Val di Fiemme MMil epo-hormooni tarvitamisega vahele jäänud Kaisa Varis põhjustas Soome suusaliidule ligi 6 miljoni Eesti krooni suuruse kahju .,"['Val di Fiemme MMil jäi Kaisa Varis epo-hormooni tarvitamisega vahele.', 'Kaisa Varis põhjustas Soome suusaliidule ligi 6 miljoni Eesti krooni suuruse kahju.']",NaN,A,a_n80,y,y
761,1013994,1614717,3,surema,NaN,adit,seenesurm,seenesurma,"Esmaspäeval suri seenesurma kaks inimest , 12 seenesõbra elu jõudsid arstid päästa .","['Esmaspäeval suri seenesurma kaks inimest.', '12 seenesõbra elu jõudsid arstid päästa.']",NaN,S,s_n80,y,y
739,938429,1494264,15,surema,NaN,adit,marutõbi,marutõppe,"Pärtel ise tegi marutaudi süstikuuri läbi kümnekonna aasta eest , kui oli paljakäsi arstinud marutõppe surnud lehma .","['Pärtel tegi marutaudi süstikuuri läbi kümnekonna aasta eest.', 'Pärtel arstis paljakäsi marutõppe surnud lehma.']",|S|AS|LS|ST|AST|LST|,NaN,s_n80,y,k
750,11322229,18146142,10,haigestuma,NaN,adit,suguhaigus,suguhaigusse,"Osa ajaloolasi arvab , et noorpõlves haigestus Friedrich II suguhaigusse , mida arstid ei suutnud ravida .","['Osa ajaloolasi arvab, et Friedrich II haigestus noorpõlves suguhaigusse.', 'Arstid ei suutnud seda haigust ravida.']",|S|AS|LS|ST|AST|LST|,NaN,s_n80,y,y
239,14279601,22668717,9,tungima,kallale,all,Dmitri,Dmitrile,Pühapäeval tungisid Narvas Rakvere tänaval kolm noormeest kallale Dmitrile .,['Pühapäeval tungisid Narvas Rakvere tänaval kolm noormeest kallale Dmitrile.'],NaN,A,a_n80,y,y


In [12]:
same_sent_alluv.sample(10)

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
228,8825925,14174174,24,kuulma,NaN,abl,Peetri,Peetrilt,"Kadumiseks oli aeg suurepärane - kui keegi hakkaks uurima , miks mind kohtus pole ja kus ma üldse olen , siis kuuleks ta Peetrilt , kuidas me koos napsi võtsime ning kuidas ma oma jala peal koju hakkasin minema .","['Kadumiseks oli aeg suurepärane.', 'Kui keegi hakkaks uurima, miks mind kohtus pole ja kus ma üldse olen, siis kuuleks ta midagi.', 'Ta kuuleks Peetrilt, kuidas me koos napsi võtsime.', 'Ta kuuleks Peetrilt, kuidas ma oma jala peal koju hakkasin minema.']",|AL|ALT|,A,a_n80,y,y
760,1201543,1910376,37,surema,NaN,adit,rasvumine,rasvumisse,"Sama kahjulikult kui nälgimine mõjub mustlaskultuurile ja mõnedele islamimaadele omane pruudi nuumamine , mistõttu perekond sai küll naise eest väärilise hinna ( kehakaal väärismetalli kilodes vm ) , kuigi õnnetu pruut ise suri noorelt südame veresoonkonna rasvumisse .","['Nälgimine mõjub mustlaskultuurile kahjulikult.', 'Nälgimine mõjub mõnedele islamimaadele omase pruudi nuumamise tõttu kahjulikult.', 'Perekond sai pruudi eest väärilise hinna.', 'Hind sõltus pruudi kehakaalust väärismetalli kilodes või muus mõõtühikus.', 'Õnnetu pruut suri noorelt südame veresoonkonna rasvumisse.']",|S|AS|LS|ST|AST|LST|,S,s_n80,y,y
253,11849203,18962736,2,mõjuma,NaN,all,aru,arule,Naisterahva arule ei mõju ükski ilm ( ühe tartlase tähelepanek ) .,['Ühe tartlase tähelepaneku järgi ei mõju ükski ilm naisterahva arule.'],NaN,NaN,a_n80,y,y
757,1801124,2866038,12,haigestuma,NaN,ill,miski,millessegi,"Üks kord narkootikumide proovimist ei tähenda veel , et ta on millessegi haigestunud .","['Üks kord narkootikumide proovimist ei tähenda veel, et ta on millessegi haigestunud.']",NaN,NaN,s_n80,y,y
790,12524189,20047313,1,haigestuma,NaN,adit,hingamisteedepõletik,Hingamisteedepõletikku,"Hingamisteedepõletikku haigestunud Vene president Boriss Jeltsin ( 67 ) jätkab Moskva-lähedases Gorki-9 residentsis ravikuuri , teatas eile presidendi pressitalitus .","['Vene president Boriss Jeltsin on haigestunud hingamisteedepõletikku.', 'Ta on 67-aastane.', 'Ta jätkab ravikuuri Moskva-lähedases Gorki-9 residentsis.', 'Presidendi pressitalitus teatas sellest eile.']",NaN,NaN,s_n80,y,y
262,9538320,15310475,11,helistama,NaN,all,Frishman,Frishmanile,"Fraktsiooni Meie Valik esimees Tatjana Muravjova ütles : “ Olen Frishmanile korduvalt helistanud ja tedaistungitele kutsunud , aga ta pole tulnud .","['Tatjana Muravjova on fraktsiooni Meie Valik esimees.', 'Tatjana Muravjova ütles midagi.', 'Tatjana Muravjova on Frishmanile korduvalt helistanud.', 'Tatjana Muravjova on Frishmanit istungitele kutsunud.', 'Frishman pole istungitele tulnud.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,a_n80,y,y
794,383545,595328,16,haigestuma,NaN,ill,veistehullus,veistehullusesse,"Lisaks sellele on USA teadlased omakorda hoiatanud , et kuigi sead , kanad ja kalad veistehullusesse ei haigestu , võivad tõvestavad prioonid nende kudedes paljuneda .","['USA teadlased on hoiatanud.', 'Sead, kanad ja kalad ei haigestu veistehullusesse.', 'Tõvestavad prioonid võivad nende kudedes paljuneda.']",|S|AS|LS|ST|AST|LST|,NaN,s_n80,y,y
740,11538931,18478063,4,haigestuma,NaN,ill,tüüfus,tüüfusesse,Kapten Šulgin haigestus tüüfusesse ning ta tuli lumele jätta .,"['Kapten Šulgin haigestus tüüfusesse.', 'Kapten Šulgin tuli lumele jätta.']",|S|AS|LS|ST|AST|LST|,NaN,s_n80,y,y
209,14253524,22626415,13,laenama,NaN,abl,iseenese,iseendilt,"Arusaamatuses ja pidetuses otsivad siis headki näitlejad laval abi stampžestidest , laenavad iseendilt .","['Arusaamatuses ja pidetuses otsivad head näitlejad laval abi stampžestidest.', 'Näitlejad laenavad iseendilt.']",NaN,A,a_n80,y,y
756,9559095,15344034,10,surema,NaN,adit,külmasurm,külmasurma,Moskva meditsiiniametnike andmeil on seal tänavuse talve algusest saadik külmasurma surnud 69 inimest .,['Moskva meditsiiniametnike andmeil on se

In [13]:
same_sent_replaced 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
237,7692281,12333295,5,näima,NaN,all,tantsuhitt,tantsuhitile,Prince'i 1982. aastal ilmunud tantsuhitile “ 1999 ” näib aastavahetusega saabuvat uus ( ja oodatud ) tulemine .,"[""Prince'i tantsuhitt “1999” ilmus 1982. aastal."", 'Aastavahetusega näib saabuvat hittile uus tulemine.', 'See tulemine on oodatud.']",NaN,NaN,a_n80,y,?
241,1892072,3014379,10,näima,NaN,all,mis,millele,"Kuigi uus ajajärk võib meid eemale juhtida kuristikust , millele Eesti näis viimastel kuudel järjest kiiremini lähenevat , jäävad mitmed ohud endiselt õhku rippuma .","['Uus ajajärk võib meid eemale juhtida kuristikust.', 'Eesti näis viimastel kuudel kuristikule järjest kiiremini lähenevat.', 'Mitmed ohud jäävad endiselt õhku rippuma.']",NaN,NaN,a_n80,y,?
264,15901007,24737765,17,kasseerima,NaN,abl,filmitäht,filmitähelt,"( Muide , kui Brigitte Bardot'tuleb selle muhameedlase käest vett ostma , kasseerib kaupmees filmitähelt kahekümnekordse hinna ! )","['Brigitte Bardot tuleb selle muhameedlase käest vett ostma.', ""Kaupmees kasseerib Brigitte Bardot'lt kahekümnekordse hinna.""]",|A|AE|AL|AS|AT|AET|ALT|AST|,A,a_n80,y,?
785,654720,1040331,19,haigestuma,NaN,ill,see,sellesse,"Ehkki rinnavähk ei ole Ameerika Ühendriikides naiste seas levinuim vähiliik ( esimesel kohal on kopsuvähk ) , haigestub sellesse iga päev 500 naist .","['Rinnavähk ei ole Ameerika Ühendriikides naiste seas levinuim vähiliik.', 'Levinuim vähiliik on kopsuvähk.', 'Iga päev haigestub rinnavähki 500 naist.']",NaN,NaN,s_n80,y,?


In [14]:
same_sent_kesksona 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
739,938429,1494264,15,surema,NaN,adit,marutõbi,marutõppe,"Pärtel ise tegi marutaudi süstikuuri läbi kümnekonna aasta eest , kui oli paljakäsi arstinud marutõppe surnud lehma .","['Pärtel tegi marutaudi süstikuuri läbi kümnekonna aasta eest.', 'Pärtel arstis paljakäsi marutõppe surnud lehma.']",|S|AS|LS|ST|AST|LST|,NaN,s_n80,y,k
745,15015961,23555547,4,haigestuma,NaN,ill,seenhaigus,seenhaigusesse,"Peale selle võib seenhaigusesse haigestunud küünest nakkus levida ka haige teistele kehaosadele , eriti kätele ja kubemevolti .","['Seenhaigusesse haigestunud küünest võib nakkus levida haige teistele kehaosadele.', 'Nakkus levib eriti kätele.', 'Nakkus levib ka kubemevolti.']",|S|AS|LS|ST|AST|LST|,NaN,s_n80,y,k
749,7487923,12017641,4,nakatuma,NaN,adit,AIDS,AIDSi,"“ Mitte ainult AIDSi nakatunud ei vaja kompleksravi : mujal maailmas lähtutakse põhimõttest , et sellega tuleb alustada nakatumise järel võimalikult ruttu , ” selgitab Raukas .","['Mitte ainult AIDSi nakatunud ei vaja kompleksravi.', 'Mujal maailmas lähtutakse põhimõttest, et kompleksravi tuleb alustada nakatumise järel võimalikult ruttu.', 'Raukas selgitab seda.']",|S|AS|LS|ST|AST|LST|,NaN,s_n80,y,k
752,1090812,1733754,1,haigestuma,NaN,adit,paragripp,Paragrippi,"Paragrippi haigestunud ja niisama nohus-köhas inimesed ostavad talvel tavapärasest rohkem vitamiine , aspiriini ning gripirohtusid , suurendades oluliselt ravimimüüjate sissetulekut .","['Paragrippi haigestunud inimesed ostavad talvel tavapärasest rohkem vitamiine, aspiriini ja gripirohtusid.', 'Nohus ja köhas inimesed ostavad talvel tavapärasest rohkem vitamiine, aspiriini ja gripirohtusid.', 'See suurendab oluliselt ravimimüüjate sissetulekut.']",|S|AS|LS|ST|AST|LST|,NaN,s_n80,y,k
753,806037,1286063,7,põdema,NaN,adit,põletik,põletikku,"Saunakuumust peaksid vältima epilepsia , peaaju põletikku põdenud ja kaugele arenenud veresoonkonna- , südame- ja neerupuudulikkusega haiged .","['Epilepsiahaiged peaksid vältima saunakuumust.', 'Peaaju põletikku põdenud haiged peaksid vältima saunakuumust.', 'Kaugele arenenud veresoonkonna puudulikkusega haiged peaksid vältima saunakuumust.', 'Kaugele arenenud südamepuudulikkusega haiged peaksid vältima saunakuumust.', 'Kaugele arenenud neerupuudulikkusega haiged peaksid vältima saunakuumust.']",|S|AS|LS|ST|AST|LST|,NaN,s_n80,y,k
786,16201132,25105865,5,nakatuma,NaN,adit,viirusartriit,viirusartriiti,Maedi Visna'sse või kitsede viirusartriiti/-entsefaliiti nakatunud loomad on,"[""Maedi Visna'sse nakatunud loomad on."", 'Kitsede viirusartriiti/-entsefaliiti nakatunud loomad on.']",NaN,NaN,s_n80,y,k
787,2514205,4031773,7,surema,NaN,adit,südametõbi,südametõppe,"Kui küüniliselt mõelda , siis iga südametõppe surnud eduka maksumaksja näol kaotab linn võrreldamatult enam , kui oleks kulunud mõne palliplatsi loomiseks .","['Kui küüniliselt mõelda, siis linn kaotab iga südametõppe surnud eduka maksumaksja näol.', 'Kaotus on võrreldamatult suurem, kui oleks kulunud mõne palliplatsi loomiseks.']",NaN,NaN,s_n80,y,k
792,14206201,22586169,3,nakatuma,NaN,adit,linnadžiibivaimustus,linnadžiibivaimustusse,"Ford tahab linnadžiibivaimustusse nakatunud Euroopas hiljemalt 2008. aastal müügile tuua väiksemat sorti nelikveolise universaalauto , millele loodetakse suuremat edu , kui praegune Maverick saavutada suutis .","['Ford tahab Euroopas müügile tuua väiksemat sorti nelikveolise universaalauto.', 'Euroopa on nakatunud linnadžiibivaimustusse.', 'Ford plaanib seda teha hiljemalt 2008. aastal.', 'Ford loodab sellele autole suuremat edu kui praegune Maverick saavutas.']",NaN,S,s_n80,y,k


In [15]:
same_sent_error 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
207,17832366,27223743,8,kinkima,NaN,all,arvuti,arvutile,ma ei kingi ju ventikat ka talle arvutile,['Ma ei kingi ventikat talle arvutile.'],NaN,NaN,a_n80,y,e
221,8969953,14421217,7,pressima,välja,abl,kaubitseja,kaubitsejatelt,"Süüdistustelaine , et turu tegevjuhid pressivad kaubitsejatelt põhjendamatult palju raha välja ja kiusavad neid taga , tõusis taas päevakorda pärast detsembri lõpus toimunud pommiplahvatust , milles hukkus üks ja sai vigastada kolm inimest .","['Turu tegevjuhte süüdistatakse kaubitsejatelt põhjendamatult palju raha välja pressimises.', 'Turu tegevjuhte süüdistatakse kaubitsejate tagakiusamises.', 'Süüdistused tõusid päevakorda pärast detsembri lõpus toimunud pommiplahvatust.', 'Pommiplahvatuses hukkus üks inimene.', 'Pommiplahvatuses sai vigastada kolm inimest.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,a_n80,y,e
272,3900385,6282179,1,nähtuma,ette,all,linnapea,Linnapeale,"Linnapeale , abilinnapeale , linnaosa vanematele ja linnasekretärile on ette nähtud 250 liitrit bensiini kuus , teistele autokasutajatele 200 liitrit .","['Linnapeale on ette nähtud 250 liitrit bensiini kuus.', 'Abilinnapeale on ette nähtud 250 liitrit bensiini kuus.', 'Linnaosa vanematele on ette nähtud 250 liitrit bensiini kuus.', 'Linnasekretärile on ette nähtud 250 liitrit bensiini kuus.', 'Teistele autokasutajatele on ette nähtud 200 liitrit bensiini kuus.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,a_n80,y,e
746,568197,898999,7,surema,NaN,adit,kant,kanti,"Rongkäigud jätkusid Rabatis kogu nädala-vahetuse , kanti surnud kuninga pilte , paljud nutsid .","['Rongkäigud jätkusid Rabatis kogu nädalavahetuse.', 'Rongkäikudel kanti surnud kuninga pilte.', 'Paljud inimesed nutsid.']",NaN,NaN,s_n80,y,e
762,12131027,19422637,4,haigestuma,NaN,adit,tiim,tiimi,"Ootamatult aga haigestus tiimi üks liige ning Treier sai bossilt telefonikõne : kui tahad meiega Prantsusmaa mitmepäevasõidul startida , ole ülehomme kohal .","['Tiimi üks liige haigestus ootamatult.', 'Treier sai bossilt telefonikõne.', 'Boss ütles, et kui Treier tahab Prantsusmaa mitmepäevasõidul startida, peab ta olema ülehomme kohal.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,s_n80,y,e
768,4703049,7556562,16,haigestuma,NaN,adit,ravimatus,ravimatusse,"Tervisekaitseamet hakkab hullulehmatõve probleemiga tegelema siis , kui esimene inimene haigestub nakatunud liha söömise tagajärjel ravimatusse Creutzfeld-Jacobi tõppe .","['Tervisekaitseamet hakkab hullulehmatõve probleemiga tegelema.', 'See juhtub siis, kui esimene inimene haigestub nakatunud liha söömise tagajärjel ravimatusse Creutzfeldt-Jakobi tõppe.']",NaN,S,s_n80,y,e
773,15364971,23997877,11,surema,NaN,adit,vang,vangi,"Usbekistanis Andijoni vanglas suri novembris piinamise tagajärjel kaks islamiäärmusluses süüdistatud vangi , ütlesid meeste sugulased ja inimõiguslased .","['Usbekistanis Andijoni vanglas suri novembris kaks vangi.', 'Vangid surid piinamise tagajärjel.', 'Mehi süüdistati islamiäärmusluses.', 'Selle ütlesid meeste sugulased ja inimõiguslased.']",NaN,A,s_n80,y,e


In [16]:
#errors

In [17]:
gpt_errors

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
247,12211695,19550360,16,uurima,NaN,abl,see,sellelt,"Miks Romeo Kalda elu läks just nii , nagu ta läinud on , uuris Toimik sellelt ühelt Eestimaa ohtlikumalt kurjategijalt endalt .","['Toimik uuris Romeo Kaldalt midagi.', 'Romeo Kalda on üks Eestimaa ohtlikumaid kurjategijaid.', 'Toimik uuris, miks Romeo Kalda elu läks nii, nagu see läinud on.']",NaN,A,a_n80,?,?
769,2089813,3334984,16,surema,NaN,adit,puudumine,puudumisse,"Elavad mõne kuu ja surevad siis madala temperatuuri , päikese või vähese söögiks kõlbliku heitvee puudumisse .","['Nad elavad mõne kuu.', 'Nad surevad madala temperatuuri tõttu.', 'Nad surevad päikese puudumise tõttu.', 'Nad surevad vähese söögiks kõlbliku heitvee puudumise tõttu.']",NaN,S,s_n80,?,?


# Kokkuvõte A ja S n80, mis on sõnastiku/gpt poolt saanud E/L/T (82 näidet)

In [9]:
same_sent = n80_elt[n80_elt["verb_head_same_sent"]=="y"]
not_same_sent = n80_elt[n80_elt["verb_head_same_sent"]=="n"]
not_same_but_error = not_same_sent[not_same_sent["head_verbi_alluv"]=="?"]
not_same_not_alluv = not_same_sent[not_same_sent["head_verbi_alluv"]=="n"]
not_same_orig_error =  not_same_sent[not_same_sent["head_verbi_alluv"]=="e"]
same_sent_alluv = same_sent[same_sent["head_verbi_alluv"]=="y"]
same_sent_replaced = same_sent[same_sent["head_verbi_alluv"]=="?"]
same_sent_kesksona = same_sent[same_sent["head_verbi_alluv"]=="k"]
same_sent_error = same_sent[same_sent["head_verbi_alluv"]=="e"]
#errors = n80_elt[n80_elt["verb_head_same_sent"]=="e"]
gpt_errors = n80_elt[n80_elt["verb_head_same_sent"]=="?"]

In [10]:
print("verb + peasõna on samas lihtlauses: ", len(same_sent))
print("verb+ps samas lihtlauses ja ps on verbi alluv: ", len(same_sent_alluv), "/", len(same_sent))
print("verb+ps samas lihtlauses aga peasõna on asendatud teise sõnaga (enamasti sisuliselt õige): ", len(same_sent_replaced), "/", len(same_sent))
print("verb+ps samas lihtlauses aga orig lauses kesksõna vorm: ", len(same_sent_kesksona), "/", len(same_sent))
print("verb+ps samas lihtlauses aga lauses probleem: ", len(same_sent_error), "/", len(same_sent))


print("\n")

print("verb + peasõna pole samas lihtlauses: ", len(not_same_sent))
print("verb+ps pole samas lihtlauses aga orig lauses on ps verbi alluv (gpt lausestamise viga):", len(not_same_but_error), "/", len(not_same_sent) )
print("verb+ps pole samas lihtlauses ja ps ei ole orig lauses verbi alluv:", len(not_same_not_alluv), "/", len(not_same_sent) )
print("verb+ps pole samas lihtlauses ja orig lauses midagi valesti:", len(not_same_orig_error), "/", len(not_same_sent) )

print("\n")

#print("peasõna süntaksi viga:", len(errors))
print("gpt lausestamisega muutus struktuur liiga palju:", len(gpt_errors))


verb + peasõna on samas lihtlauses:  76
verb+ps samas lihtlauses ja ps on verbi alluv:  59 / 76
verb+ps samas lihtlauses aga peasõna on asendatud teise sõnaga (enamasti sisuliselt õige):  1 / 76
verb+ps samas lihtlauses aga orig lauses kesksõna vorm:  0 / 76
verb+ps samas lihtlauses aga lauses probleem:  16 / 76


verb + peasõna pole samas lihtlauses:  5
verb+ps pole samas lihtlauses aga orig lauses on ps verbi alluv (gpt lausestamise viga): 2 / 5
verb+ps pole samas lihtlauses ja ps ei ole orig lauses verbi alluv: 0 / 5
verb+ps pole samas lihtlauses ja orig lauses midagi valesti: 3 / 5


gpt lausestamisega muutus struktuur liiga palju: 1


### Segadusmaatriks

In [15]:
sm_elt = n80_elt[n80_elt["verb_head_same_sent"]!= "?"]
sm_elt.loc[
    (sm_elt["verb_head_same_sent"] == "n") &
    (sm_elt["head_verbi_alluv"] == "?"),
    "head_verbi_alluv"
] = "y"
sm_elt = sm_elt[(sm_elt["head_verbi_alluv"]!= "?")]

In [16]:
pd.crosstab(
    sm_elt["verb_head_same_sent"],
    sm_elt["head_verbi_alluv"],
    rownames=["SameSent"],
    colnames=["Alluv"]
)

Alluv,e,y
SameSent,,
n,3,2
y,16,59


### Näited

In [9]:
not_same_sent 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
225,11072933,17736942,5,märkima,NaN,all,Eesti,Eestile,"Käesolev aasta on nii Eestile , Lätile kui ka Leedule alles esimene hooaeg NATO liikmelisuse saavutamise aastaprogrammi ( membership action plan ehk MAP - toim ) raames , märkis peasekretär .","['Käesolev aasta on Eestile esimene hooaeg NATO liikmelisuse saavutamise aastaprogrammi raames.', 'Käesolev aasta on Lätile esimene hooaeg NATO liikmelisuse saavutamise aastaprogrammi raames.', 'Käesolev aasta on Leedule esimene hooaeg NATO liikmelisuse saavutamise aastaprogrammi raames.', 'Peasekretär märkis seda.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,a_n80,n,e
245,2342833,3753052,2,kompenseerima,NaN,all,seadus,seadusele,"Vastavalt seadusele kompenseerib kohalik omavalitsus investeeringud teedeehitusse , drenaaži ja tänavavalgutusse , juhul kui need on avalikus kasutuses või kui ei lepita kokku teisiti .","['Seadus sätestab kohaliku omavalitsuse kohustused.', 'Kohalik omavalitsus kompenseerib investeeringud teedeehitusse, drenaaži ja tänavavalgustusse.', 'Kompensatsioon toimub juhul, kui need on avalikus kasutuses.', 'Kompensatsioon toimub ka juhul, kui ei lepita kokku teisiti.']",NaN,NaN,a_n80,n,e
267,3329176,5354106,4,sätestama,NaN,all,toode,toodetele,"Seadus sätestab teatud toodetele , nagu mootorsõidukid ja nende osad , tootja vastutuse põhimõtte .","['Seadus sätestab tootja vastutuse põhimõtte.', 'See kehtib teatud toodetele.', 'Need tooted on näiteks mootorsõidukid ja nende osad.']",NaN,L,a_n80,n,?
330,5546970,8894196,10,laenama,NaN,abl,klubi,klubilt,"Mart Poomi koduklubi Derby County laenas algavaks hooajaks Itaalia klubilt AC Milan kaitsemängija Daniele Daino , teatas Sportnet .","['Mart Poomi koduklubi on Derby County.', 'Derby County laenas algavaks hooajaks kaitsemängija Daniele Daino.', 'Daniele Daino kuulub Itaalia klubile AC Milan.', 'Sportnet teatas sellest.']",NaN,L,a_n80,n,?
347,8824416,14171424,3,ütlema,ära,all,töö,töödele,Ütlesin siiski töödele ja kohtumistele viidates viisakalt ära .,"['Ütlesin viisakalt ära.', 'Viitasin töödele ja kohtumistele.']",NaN,L,a_n80,n,e
776,16334815,25281664,61,surema,NaN,adit,kanza-surm,kanza-surma,"niisiis , uurisin antud linki sealt lehelt veel linke ja otsisin veel täiendust ja sain päris palju teada selle asja kohta : esiteks final fantasy sulle teadmiseks et alkohol seejärel tubakas on palju ohtlikumad ja suuremat sõltuvust tekitavad asjad kui kanza ( kanep tekitab vähem sõltuvust kui kohvi ntx ) , kanepiga pole võimalik üledoosi kätte surra , keegi pole kanza-surma surnud , kanza ei tekita inimeses vastupidiselt ntx muudele narkodele ja alcole vägivallatunnet , pigem rahulikku ja mõnusa olemise ( nagu mõne pilsneri joomine ) , kanza on paljus mõttes kasulik ( saab teha köit , ravimina jnejne . )","['Uurisin antud linki.', 'Leidsin sealt lehelt veel linke.', 'Otsisin veel täiendust.', 'Sain päris palju teada selle asja kohta.', 'Final Fantasy, sulle teadmiseks, et alkohol ja tubakas on palju ohtlikumad kui kanep.', 'Kanep tekitab vähem sõltuvust kui kohv.', 'Kanepiga pole võimalik üledoosi kätte surra.', 'Keegi pole kanepitarbimise tõttu surnud.', 'Kanep ei tekita inimeses vägivallatunnet.', 'Kanep tekitab pigem rahuliku ja mõnusa olemise.', 'Kanep on paljus mõttes kasulik.', 'Kanepist saab teha köit.', 'Kanepit kasutatakse ravimina.']",NaN,NaN,s_n80,n,?


In [10]:
not_same_but_error

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
267,3329176,5354106,4,sätestama,NaN,all,toode,toodetele,"Seadus sätestab teatud toodetele , nagu mootorsõidukid ja nende osad , tootja vastutuse põhimõtte .","['Seadus sätestab tootja vastutuse põhimõtte.', 'See kehtib teatud toodetele.', 'Need tooted on näiteks mootorsõidukid ja nende osad.']",NaN,L,a_n80,n,?
330,5546970,8894196,10,laenama,NaN,abl,klubi,klubilt,"Mart Poomi koduklubi Derby County laenas algavaks hooajaks Itaalia klubilt AC Milan kaitsemängija Daniele Daino , teatas Sportnet .","['Mart Poomi koduklubi on Derby County.', 'Derby County laenas algavaks hooajaks kaitsemängija Daniele Daino.', 'Daniele Daino kuulub Itaalia klubile AC Milan.', 'Sportnet teatas sellest.']",NaN,L,a_n80,n,?
776,16334815,25281664,61,surema,NaN,adit,kanza-surm,kanza-surma,"niisiis , uurisin antud linki sealt lehelt veel linke ja otsisin veel täiendust ja sain päris palju teada selle asja kohta : esiteks final fantasy sulle teadmiseks et alkohol seejärel tubakas on palju ohtlikumad ja suuremat sõltuvust tekitavad asjad kui kanza ( kanep tekitab vähem sõltuvust kui kohvi ntx ) , kanepiga pole võimalik üledoosi kätte surra , keegi pole kanza-surma surnud , kanza ei tekita inimeses vastupidiselt ntx muudele narkodele ja alcole vägivallatunnet , pigem rahulikku ja mõnusa olemise ( nagu mõne pilsneri joomine ) , kanza on paljus mõttes kasulik ( saab teha köit , ravimina jnejne . )","['Uurisin antud linki.', 'Leidsin sealt lehelt veel linke.', 'Otsisin veel täiendust.', 'Sain päris palju teada selle asja kohta.', 'Final Fantasy, sulle teadmiseks, et alkohol ja tubakas on palju ohtlikumad kui kanep.', 'Kanep tekitab vähem sõltuvust kui kohv.', 'Kanepiga pole võimalik üledoosi kätte surra.', 'Keegi pole kanepitarbimise tõttu surnud.', 'Kanep ei tekita inimeses vägivallatunnet.', 'Kanep tekitab pigem rahuliku ja mõnusa olemise.', 'Kanep on paljus mõttes kasulik.', 'Kanepist saab teha köit.', 'Kanepit kasutatakse ravimina.']",NaN,NaN,s_n80,n,?


In [11]:
not_same_not_alluv

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv


In [12]:
not_same_orig_error

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
225,11072933,17736942,5,märkima,NaN,all,Eesti,Eestile,"Käesolev aasta on nii Eestile , Lätile kui ka Leedule alles esimene hooaeg NATO liikmelisuse saavutamise aastaprogrammi ( membership action plan ehk MAP - toim ) raames , märkis peasekretär .","['Käesolev aasta on Eestile esimene hooaeg NATO liikmelisuse saavutamise aastaprogrammi raames.', 'Käesolev aasta on Lätile esimene hooaeg NATO liikmelisuse saavutamise aastaprogrammi raames.', 'Käesolev aasta on Leedule esimene hooaeg NATO liikmelisuse saavutamise aastaprogrammi raames.', 'Peasekretär märkis seda.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,a_n80,n,e
245,2342833,3753052,2,kompenseerima,NaN,all,seadus,seadusele,"Vastavalt seadusele kompenseerib kohalik omavalitsus investeeringud teedeehitusse , drenaaži ja tänavavalgutusse , juhul kui need on avalikus kasutuses või kui ei lepita kokku teisiti .","['Seadus sätestab kohaliku omavalitsuse kohustused.', 'Kohalik omavalitsus kompenseerib investeeringud teedeehitusse, drenaaži ja tänavavalgustusse.', 'Kompensatsioon toimub juhul, kui need on avalikus kasutuses.', 'Kompensatsioon toimub ka juhul, kui ei lepita kokku teisiti.']",NaN,NaN,a_n80,n,e
347,8824416,14171424,3,ütlema,ära,all,töö,töödele,Ütlesin siiski töödele ja kohtumistele viidates viisakalt ära .,"['Ütlesin viisakalt ära.', 'Viitasin töödele ja kohtumistele.']",NaN,L,a_n80,n,e


In [13]:
same_sent.sample(5)

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
785,654720,1040331,19,haigestuma,NaN,ill,see,sellesse,"Ehkki rinnavähk ei ole Ameerika Ühendriikides naiste seas levinuim vähiliik ( esimesel kohal on kopsuvähk ) , haigestub sellesse iga päev 500 naist .","['Rinnavähk ei ole Ameerika Ühendriikides naiste seas levinuim vähiliik.', 'Levinuim vähiliik on kopsuvähk.', 'Iga päev haigestub rinnavähki 500 naist.']",NaN,NaN,s_n80,y,?
777,2626340,4214204,7,surema,NaN,adit,polooniumimürgitus,polooniumimürgitusse,Litvinenko suri eelmise aasta novembris Londonis polooniumimürgitusse .,['Litvinenko suri eelmise aasta novembris Londonis polooniumimürgitusse.'],NaN,E,s_n80,y,y
386,10556686,16921384,1,lugema,NaN,all,maakataster,Maakatastrile,"Maakatastrile loeb see , milline krunt on enne katastrisse kantud - selle järgi mõõdetakse välja külgnev maa-ala , ehkki piirid ei pruugi olla täiesti täpsed .","['Maakatastrile loeb, milline krunt on enne katastrisse kantud.', 'Selle järgi mõõdetakse välja külgnev maa-ala.', 'Piirid ei pruugi olla täiesti täpsed.']",NaN,L,a_n80,y,y
295,4963673,7969629,20,toppima,NaN,all,mängumaa,mängumaale,"Otsuse , kes seal on teeb Isamaaliit ja kuni ei ole tegemist tõsise kuritahtlusega Reformierakond oma nina teiste erakondade mängumaale ei topi .","['Otsuse, kes seal on, teeb Isamaaliit.', 'Reformierakond ei topi oma nina teiste erakondade mängumaale.', 'Reformierakond ei topi oma nina, kui ei ole tegemist tõsise kuritahtlusega.']",NaN,L,a_n80,y,y
200,11407948,18281923,24,jääma,võlgu,all,liit,Liidule,"Jutt on umbes 2-2 , 5 miljardist dollarist , mis kukutatud Mengistu Haile Mariami režiim jäi just saadud sõjavarustuse eest võlgu veel Nõukogude Liidule .","['Jutt on umbes 2-2,5 miljardist dollarist.', 'Kukutatud Mengistu Haile Mariami režiim jäi selle summa sõjavarustuse eest võlgu Nõukogude Liidule.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,a_n80,y,y


In [14]:
same_sent_alluv.sample(10)

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
211,6195531,9948684,10,sõnama,NaN,all,portaal,portaalile,"Aeg-ajalt käikase üle lahe lennumasinatega , "" sõnas ta portaalile .","['Aeg-ajalt käiakse üle lahe lennumasinatega.', 'Ta sõnas seda portaalile.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,a_n80,y,y
793,2676753,4294081,5,nakatuma,NaN,adit,taud,taudi,Waugh' sead nakatusid taudi ilmselt toidujäätmeid süües .,"[""Waugh' sead nakatusid taudi."", 'Sead nakatusid ilmselt toidujäätmeid süües.']",NaN,NaN,s_n80,y,y
742,9921525,15917426,27,haigestuma,NaN,adit,ALD-tõbi,ALD-tõppe,"Lorenzo on Ameerika itaallaste Michaela ( Susan Sarandon ) ja Augusto ( Nick Nolte ) Odonede ainus laps , kes ühel päeval haigestub piinava surmaga lõppevasse ALD-tõppe , mida tuntakse ka adrenoleukodüstroofia nime all .","['Lorenzo on Michaela ja Augusto Odonede ainus laps.', 'Michaela ja Augusto Odone on Ameerika itaallased.', 'Lorenzo haigestus piinava surmaga lõppevasse ALD-tõppe.', 'ALD-tõbe tuntakse ka adrenoleukodüstroofia nime all.']",NaN,NaN,s_n80,y,y
771,741926,1182159,8,surema,NaN,adit,vanadus,vanadusse,"Meil olid lõvid , aga nad surid vanadusse ja me plaanime tuua siia aasia lõvid .","['Meil olid lõvid.', 'Lõvid surid vanadusse.', 'Me plaanime tuua siia Aasia lõvid.']",NaN,T,s_n80,y,y
608,18064302,27483889,16,surema,NaN,adit,säng,sängi,"Nemad on kõige vajalikumad ja altruistlikumad virulased , kes surevad tavaliselt pärast rasket elutoimetust oma sängi , ja nende najal , võite kindlad olla , püsib Virumaa , püsib Eesti riik .","['Nemad on kõige vajalikumad virulased.', 'Nemad on kõige altruistlikumad virulased.', 'Nemad surevad tavaliselt pärast rasket elutoimetust oma sängi.', 'Nende najal püsib Virumaa.', 'Nende najal püsib Eesti riik.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,s_n80,y,y
361,5036830,8084437,15,kinkima,NaN,all,jalgpallimeeskond,jalgpallimeeskonnale,Saudi Araabia miljardärist prints Al-Walid bin Talal kinkis Araabia superkarikasarjas esimeseks ja teiseks tulnud jalgpallimeeskonnale 113 luksusautot .,"['Saudi Araabia prints Al-Walid bin Talal on miljardär.', 'Prints kinkis Araabia superkarikasarjas esimeseks tulnud jalgpallimeeskonnale luksusautosid.', 'Prints kinkis Araabia superkarikasarjas teiseks tulnud jalgpallimeeskonnale luksusautosid.', 'Kokku kinkis prints 113 luksusautot.']",NaN,L,a_n80,y,y
782,6158711,9885944,26,nakatuma,NaN,adit,tüvi,tüvve,"NEW YORK , 30. märts ( AP-EPLO ) - Tervishoiuametnikud on tuvastanud juba mitmeid patsiente , kes on ilmselt nakatunud harvaesinevasse ja ravimitele allumatusse HI-viiruse tüvve , kuid pole selge , kas need juhtumid on seotud .","['NEW YORK, 30. märts (AP-EPLO) - Tervishoiuametnikud on tuvastanud mitmeid patsiente.', 'Need patsiendid on ilmselt nakatunud harvaesinevasse HI-viiruse tüvve.', 'See tüvi ei allu ravimitele.', 'Pole selge, kas need juhtumid on omavahel seotud.']",NaN,NaN,s_n80,y,y
632,9329572,14985266,5,nakatuma,NaN,adit,Hip-Hop,Hip-Hopi,Olen nakatunud Prantsuse Kuninglikku Hip-Hopi .,['Olen nakatunud Prantsuse Kuninglikku Hip-Hopi.'],|L|AL|EL|LS|LT|ALT|ELT|LST|,L,s_n80,y,y
789,2871850,4604914,6,haigestuma,NaN,ill,mis,millesse,"Need on ka vähihaigused , millesse mehed enim haigestuvad .","['Need on ka vähihaigused.', 'Need on vähihaigused, millesse mehed enim haigestuvad.']",NaN,NaN,s_n80,y,y
204,6485006,10429095,22,kehtima,NaN,all,Iraan,Iraanile,"Ma tean , et täieliku kütusetsükli hankimine tähendab , et riik on mõne kuu kaugusel tuumarelvadest , ja see kehtib nii Iraanile kui kõigile teistele .","['Ma tean, et täieliku kütusetsükli hankimine tähendab, et riik on mõne kuu kaugusel tuumarelvadest.', 'See kehtib nii Iraanile kui kõigile teistele.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,a_n80,y,y


In [15]:
same_sent_replaced 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
237,7692281,12333295,5,näima,NaN,all,tantsuhitt,tantsuhitile,Prince'i 1982. aastal ilmunud tantsuhitile “ 1999 ” näib aastavahetusega saabuvat uus ( ja oodatud ) tulemine .,"[""Prince'i tantsuhitt “1999” ilmus 1982. aastal."", 'Aastavahetusega näib saabuvat hittile uus tulemine.', 'See tulemine on oodatud.']",NaN,NaN,a_n80,y,?
241,1892072,3014379,10,näima,NaN,all,mis,millele,"Kuigi uus ajajärk võib meid eemale juhtida kuristikust , millele Eesti näis viimastel kuudel järjest kiiremini lähenevat , jäävad mitmed ohud endiselt õhku rippuma .","['Uus ajajärk võib meid eemale juhtida kuristikust.', 'Eesti näis viimastel kuudel kuristikule järjest kiiremini lähenevat.', 'Mitmed ohud jäävad endiselt õhku rippuma.']",NaN,NaN,a_n80,y,?
391,14263201,22641300,8,laulma,NaN,all,elamu,elamule,"Üle viigipükste ääre valguva kõhukese võbisedes laulab elamule ülistuslaulu ja nendib ohates , et noortel on tänapäeval tõesti raske elupinda leida .","['Üle viigipükste ääre valguv kõhuke võbiseb.', 'Kõhuke laulab elamule ülistuslaulu.', 'Kõhuke nendib ohates, et noortel on tänapäeval raske elupinda leida.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,a_n80,y,?
785,654720,1040331,19,haigestuma,NaN,ill,see,sellesse,"Ehkki rinnavähk ei ole Ameerika Ühendriikides naiste seas levinuim vähiliik ( esimesel kohal on kopsuvähk ) , haigestub sellesse iga päev 500 naist .","['Rinnavähk ei ole Ameerika Ühendriikides naiste seas levinuim vähiliik.', 'Levinuim vähiliik on kopsuvähk.', 'Iga päev haigestub rinnavähki 500 naist.']",NaN,NaN,s_n80,y,?


In [16]:
same_sent_kesksona 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
786,16201132,25105865,5,nakatuma,NaN,adit,viirusartriit,viirusartriiti,Maedi Visna'sse või kitsede viirusartriiti/-entsefaliiti nakatunud loomad on,"[""Maedi Visna'sse nakatunud loomad on."", 'Kitsede viirusartriiti/-entsefaliiti nakatunud loomad on.']",NaN,NaN,s_n80,y,k
787,2514205,4031773,7,surema,NaN,adit,südametõbi,südametõppe,"Kui küüniliselt mõelda , siis iga südametõppe surnud eduka maksumaksja näol kaotab linn võrreldamatult enam , kui oleks kulunud mõne palliplatsi loomiseks .","['Kui küüniliselt mõelda, siis linn kaotab iga südametõppe surnud eduka maksumaksja näol.', 'Kaotus on võrreldamatult suurem, kui oleks kulunud mõne palliplatsi loomiseks.']",NaN,NaN,s_n80,y,k


In [17]:
same_sent_error 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
207,17832366,27223743,8,kinkima,NaN,all,arvuti,arvutile,ma ei kingi ju ventikat ka talle arvutile,['Ma ei kingi ventikat talle arvutile.'],NaN,NaN,a_n80,y,e
210,12407194,19864326,7,tunnistama,NaN,all,kriisiõppus,kriisiõppusele,Teatrijuhtimise praktika tunnistab ta just tänu kriisiõppusele tugevaks .,"['Teatrijuhtimise praktika on tugev.', 'Ta tunnistab teatrijuhtimise praktika tugevaks tänu kriisiõppusele.']",NaN,E,a_n80,y,e
274,9361911,15036299,1,pakutama,NaN,all,investeering,Investeeringutele,Investeeringutele pakutavad maksuvabadused on pannud riigid eriti teravalt konkureerima kapitalimaksude rindel .,"['Investeeringutele pakutavad maksuvabadused on pannud riigid konkureerima kapitalimaksude rindel.', 'Riigid konkureerivad kapitalimaksude rindel eriti teravalt.']",NaN,L,a_n80,y,e
291,10000631,16042033,9,meenutama,NaN,all,tee,teedele,"Auklikule Pärnu maanteele jõudes meenutasid soomlased reise Venemaa teedele , mis klopib peaaegu hambad suust välja .","['Soomlased jõudsid auklikule Pärnu maanteele.', 'Soomlased meenutasid reise Venemaa teedele.', 'Venemaa teed klopivad peaaegu hambad suust välja.']",NaN,L,a_n80,y,e
302,18553232,28083547,1,soovitama,NaN,all,kõvaketas,Kõvakettale,"Kõvakettale , kus Linux ise asub , ei soovita seda panna .","['Linux asub kõvakettal.', 'Seda ei soovitata panna kõvakettale, kus Linux asub.']",NaN,L,a_n80,y,e
305,9012778,14490858,25,tasuma,NaN,all,majandussuhted,majandussuhetele,"Äsja Mordvast naasnud Eesti Moskva-suursaadik Tiit Matsulevitsh arvab , et Eesti ei peaks suhtluses soome-ugri sugulasrahvastega piirduma vaid kultuurikoostööga , vaid mõelda tasuks ka majandussuhetele .","['Tiit Matsulevitsh on Eesti Moskva-suursaadik.', 'Tiit Matsulevitsh naasis äsja Mordvast.', 'Tiit Matsulevitsh arvab, et Eesti ei peaks suhtluses soome-ugri sugulasrahvastega piirduma ainult kultuurikoostööga.', 'Tiit Matsulevitsh arvab, et tasuks mõelda ka majandussuhetele.']",NaN,L,a_n80,y,e
612,5654815,9068765,12,nakatuma,NaN,adit,veresoon,veresoonde,1 206 HIV juhtumist 71% ehk 836 inimest nakatusid HIVsse uimastite süstimisega veresoonde .,['1 206 HIV juhtumist 71% ehk 836 inimest nakatusid HIVsse uimastite süstimisega veresoonde.'],NaN,L,s_n80,y,e
630,867688,1382145,2,surema,NaN,adit,ajakiri,ajakirja,Suri ajakirja Novõi Mir peatoimetaja,['Suri ajakirja Novõi Mir peatoimetaja.'],NaN,L,s_n80,y,e
635,6364090,10226621,15,haigestuma,NaN,adit,kodu,koju,"Tegemist oli kaks aastat Hiinas Guandongi provintsis töötanud 33aastase mehega , kes haigestus teel koju .","['Tegemist oli 33-aastase mehega.', 'Mees töötas kaks aastat Hiinas Guandongi provintsis.', 'Mees haigestus teel koju.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,s_n80,y,e
658,1963565,3131233,13,surema,NaN,adit,veen,veeni,"Kaheksa ühe Bangladeshi haigla patsienti surid pärast vananenuks osutunud füsioloogilise lahuse manustamist veeni , 15 inimesel aga tekkis kõrge palavik .","['Kaheksa ühe Bangladeshi haigla patsienti surid.', 'Patsiendid surid pärast vananenuks osutunud füsioloogilise lahuse manustamist veeni.', '15 inimesel tekkis kõrge palavik.']",NaN,L,s_n80,y,e


In [29]:
#errors

In [18]:
gpt_errors

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
784,6215791,9983047,5,surema,NaN,adit,sketš,sketši,"Kui parafraseerida Monty Pythoni sketši surnud papagoist , siis tuleb Tony Blair Tallinna sõnumiga "" It is not dead , it's merely resting "" - s.t eesistumisel pole häda midagi , tuleb vaid oodata .","['Tony Blair tuleb Tallinna sõnumiga.', 'Sõnum on ""It is not dead, it\'s merely resting"".', 'See tähendab, et eesistumisel pole häda midagi.', 'Tuleb vaid oodata.']",NaN,L,s_n80,?,?
